# Distributed PyTorch training with Ray Train — official Ray Train DLC

The same `train.py` as the homogeneous notebook, scaled out across a heterogeneous cluster: a CPU-only coordinator head plus a group of GPU workers, running on AWS's official **Ray Train Deep Learning Container** (`public.ecr.aws/deep-learning-containers/ray:train-ml-cuda-v1.1`) — see [`../README.md`](../README.md) for details. The difference from the homogeneous notebook is entirely in the cluster configuration below — the training code does not change.

***

## Prerequisites

In [ ]:
%pip install -r ./scripts/requirements.txt --upgrade

In [ ]:
# Copy the shared Ray launcher script into ./scripts (not committed here).
%cp ../../../scripts/launcher.py ./scripts/

***

# Step 1 - Session and role

In [ ]:
from sagemaker.core.helper.session_helper import get_execution_role, Session

In [ ]:
sagemaker_session = Session()

bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix
role = get_execution_role()

# Step 2 - Resolve the Ray Train DLC image

AWS also publishes this image directly in a private, per-region ECR account under repository `ray` — the same distribution mechanism every other DLC (`pytorch-training`, `huggingface-training`, ...) already uses, with the same broad cross-account pull permissions. `sagemaker:CreateTrainingJob` pulls from there directly; `ray_dlc_image.get_ray_train_dlc_image_uri()` just resolves the right URI for your region — see [`../README.md`](../README.md) for how this was confirmed.

In [ ]:
import sys
sys.path.insert(0, "..")
from ray_dlc_image import get_ray_train_dlc_image_uri

image_uri = get_ray_train_dlc_image_uri(sagemaker_session.boto_session.region_name)
image_uri

# Step 3 - Launch the training job (heterogeneous cluster)

A CPU-only coordinator head (`ml.t3.2xlarge`, `head_num_cpus/gpus=0`) plus a GPU worker group of `4x ml.g5.xlarge`. `train.py` auto-sizes `num_workers` to the 4 worker GPUs with no code change.

**Note:** the Ray Train DLC bakes `FI_PROVIDER=efa` into its image `ENV` unconditionally. `launcher.py` treats any pre-existing value as an explicit override and skips its own EFA-device autodetection — harmless here since neither `ml.t3.2xlarge` nor `ml.g5.xlarge` has EFA hardware to conflict with, but worth knowing if you swap in an EFA-capable instance group (see [`../README.md`](../README.md)).

In [ ]:
! pygmentize ./scripts/train.py

In [ ]:
from sagemaker.train.configs import (
    CheckpointConfig,
    Compute,
    InstanceGroup,
    OutputDataConfig,
    RemoteDebugConfig,
    SourceCode,
    StoppingCondition,
)
from sagemaker.train.model_trainer import ModelTrainer

instance_groups = [
    InstanceGroup(
        instance_group_name="head-instance-group",
        instance_type="ml.t3.2xlarge",  # CPU-only coordinator
        instance_count=1,
    ),
    InstanceGroup(
        instance_group_name="worker-instance-group",
        instance_type="ml.g5.xlarge",   # GPU workers
        instance_count=4,
    ),
]

args = [
    "-e",
    "train.py",
    "--epochs",
    "20",
    "--learning_rate",
    "0.001",
    "--batch_size",
    "128",
]

source_code = SourceCode(
    source_dir="./scripts",
    requirements="requirements.txt",
    command=f"python launcher.py {' '.join(args)}",
)

compute_configs = Compute(
    instance_groups=instance_groups,
    keep_alive_period_in_seconds=0,
)

job_name = "train-ray-train-cifar10-het"
if default_prefix:
    output_path = f"s3://{bucket_name}/{default_prefix}/{job_name}"
else:
    output_path = f"s3://{bucket_name}/{job_name}"

model_trainer = ModelTrainer(
    training_image=image_uri,
    source_code=source_code,
    base_job_name=job_name,
    compute=compute_configs,
    stopping_condition=StoppingCondition(max_runtime_in_seconds=18000),
    output_data_config=OutputDataConfig(
        s3_output_path=output_path, compression_type="NONE"
    ),
    checkpoint_config=CheckpointConfig(
        s3_uri=output_path + "/checkpoints", local_path="/opt/ml/checkpoints"
    ),
    environment={
        "head_instance_group": "head-instance-group",  # which group is the Ray head
        "head_num_cpus": "0",  # coordinator-only head
        "head_num_gpus": "0",
    },
    role=role,
).with_remote_debug_config(RemoteDebugConfig(enable_remote_debug=True))

In [ ]:
# Start the training job.
model_trainer.train(wait=False)

The coordinator-only head runs no training; the 4 GPU workers do the data-parallel work and checkpoint to S3 via `CheckpointConfig`.

**Note:** `train.py`'s dataset download (torchvision's default CIFAR-10 mirror) can be slow depending on network path — give the job enough time (or stage the dataset via an S3 input channel instead) to reliably reach a completed epoch and checkpoint. See [`../README.md`](../README.md).